In [ ]:
# ros2 launch dsr_bringup2 dsr_bringup2_rviz.launch.py mode:=real host:=110.120.1.68 model:=e0509
# ros2 run dsr_gripper gripper_service

In [2]:
import rclpy
import DR_init

ROBOT_ID = "dsr01"  #
ROBOT_MODEL = "e0509"  #

DR_init.__dsr__id = ROBOT_ID  #
DR_init.__dsr__model = ROBOT_MODEL  #

rclpy.init()  #
node = rclpy.create_node("doosan_teaching_node", namespace=ROBOT_ID)  #
DR_init.__dsr__node = node  #

from DSR_ROBOT2 import (  #[cite: 1]
    set_robot_mode, get_robot_mode, get_current_posx, wait,
    ROBOT_MODE_MANUAL, ROBOT_MODE_AUTONOMOUS
)

print("✅ 초기화 완료. 티칭 단계를 진행하세요.")

_robot_id =dsr01
_robot_model =e0509
_srv_name_prefix =dsr_controller2/
_topic_name_prefix =dsr_controller2/
✅ 초기화 완료. 티칭 단계를 진행하세요.


In [6]:
# 수동(manual) 모드로 전환
set_robot_mode(ROBOT_MODE_MANUAL)  #[cite: 1]
wait(1)  #[cite: 1]

print(f"현재 로봇 모드: {get_robot_mode()} (0이면 manual 성공)")  #[cite: 1]
print("👉 로봇의 직접교시 버튼을 누르고 [Pick 위치]로 팔을 이동시킨 후 다음 셀을 실행하세요.")  #[cite: 1]

현재 로봇 모드: 0 (0이면 manual 성공)
👉 로봇의 직접교시 버튼을 누르고 [Pick 위치]로 팔을 이동시킨 후 다음 셀을 실행하세요.


In [4]:
# 현재의 TCP 좌표 [x, y, z, rx, ry, rz] 읽기
current_x, _ = get_current_posx()  #[cite: 1]

# 소수점 셋째 자리까지 반올림하여 보기 좋게 기록하고 변수에 저장
PICK_POS = [round(v, 3) for v in current_x]

print("💾 [Pick 위치 저장 완료]")
print("👉 PICK_POS =", PICK_POS)

💾 [Pick 위치 저장 완료]
👉 PICK_POS = [330.492, 185.379, 524.939, 154.139, -178.609, 66.843]


In [5]:
# 현재의 TCP 좌표 [x, y, z, rx, ry, rz] 읽기
current_x, _ = get_current_posx()  #[cite: 1]

# 소수점 셋째 자리까지 반올림하여 보기 좋게 기록하고 변수에 저장
PICK_POS = [round(v, 3) for v in current_x]

print("💾 [Pick 위치 저장 완료]")
print("👉 PICK_POS =", PICK_POS)

💾 [Pick 위치 저장 완료]
👉 PICK_POS = [315.62, 169.063, 209.479, 96.154, 177.977, 12.187]


In [6]:
# 현재의 TCP 좌표 [x, y, z, rx, ry, rz] 읽기
current_x, _ = get_current_posx()  #[cite: 1]

# 소수점 셋째 자리까지 반올림하여 보기 좋게 기록하고 변수에 저장
PICK_POS = [round(v, 3) for v in current_x]

print("💾 [Pick 위치 저장 완료]")
print("👉 PICK_POS =", PICK_POS)

💾 [Pick 위치 저장 완료]
👉 PICK_POS = [324.159, -208.944, 199.813, 112.0, -176.811, 28.156]


그리퍼 API 준비 완료


In [13]:
from DSR_ROBOT2 import posx, posj, movel, movej, wait, DR_BASE

# 1. 속도 설정 (안전 기동 속도)
SPEED_L = 60  # mm/s
ACC_L = 60

# 2. X, Y, Z축 매칭을 완료한 정렬 좌표 리스트
PICK_1 = [330.492, 185.379, 524.939, 154.139, -178.609, 66.843]
PICK_2 = [330.492, 185.379, 209.479, 154.139, -178.609, 66.843]  # Z만 변경
PICK_3 = [324.159, -208.944, 524.939, 154.139, -178.609, 66.843] # Z는 1번 유지, X/Y는 4번 일치
PICK_4 = [324.159, -208.944, 199.813, 154.139, -178.609, 66.843] # X/Y는 3번 유지, Z만 변경

print("✨ Y축 이동 및 상공 대기 좌표(3번) 세팅이 완료되었습니다.")

✨ Y축 이동 및 상공 대기 좌표(3번) 세팅이 완료되었습니다.


In [11]:
from dsr_gripper import gripper_open, gripper_close, gripper_cmd
print("그리퍼 API 준비 완료")

그리퍼 API 준비 완료


In [14]:
try:
    
    print("그리퍼 열기")
    gripper_open()
    wait(1)
    
    print("🚀 [시작] 1번 기본 위치로 기동")
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    # --- 1 ➡️ 2 ➡️ 1 구간 (수직 하강 및 상승) ---
    print("🚀 [1 ➡️ 2] 수직 하강 (Z축만 이동)")
    movel(posx(PICK_2), vel=SPEED_L * 0.5, acc=ACC_L * 0.5, ref=DR_BASE) # 진입 시 감속
    wait(1)
    
    print("그리퍼 상장 폭만큼 힘으로 닫기")
    gripper_close(current=50)
    wait(3)


    print("🚀 [2 ➡️ 1] 수직 상승 복귀")
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    # --- 1 ➡️ 3 ➡️ 4 구간 (X, Y축 평면 이동 후 수직 하강) ---
    print("🚀 [1 ➡️ 3] 수평 평면 이동 (X, Y축 동시 이동하여 4번 상공 안착)")
    movel(posx(PICK_3), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    print("🚀 [3 ➡️ 4] 4번 타겟으로 수직 하강")
    movel(posx(PICK_4), vel=SPEED_L * 0.5, acc=ACC_L * 0.5, ref=DR_BASE) # 진입 시 감속
    wait(1)

    print("그리퍼 열기")
    gripper_open()
    wait(3)
    
    # --- 4 ➡️ 3 ➡️ 1 구간 (역순 안전 복귀) ---
    print("🚀 [4 ➡️ 3] 3번 상공으로 수직 상승")
    movel(posx(PICK_3), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    print("🚀 [3 ➡️ 1] 다시 1번 상공으로 수평 복귀 (X, Y축 복귀)")
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    print("🎉 [성공] Y축 피드백을 반영하여 직각 및 평면 모션을 안전하게 완수했습니다!")

except Exception as e:
    print(f"❌ 구동 중 에러 발생: {e}")

그리퍼 열기
🚀 [시작] 1번 기본 위치로 기동
🚀 [1 ➡️ 2] 수직 하강 (Z축만 이동)
그리퍼 상장 폭만큼 힘으로 닫기
🚀 [2 ➡️ 1] 수직 상승 복귀
🚀 [1 ➡️ 3] 수평 평면 이동 (X, Y축 동시 이동하여 4번 상공 안착)
🚀 [3 ➡️ 4] 4번 타겟으로 수직 하강
그리퍼 열기
🚀 [4 ➡️ 3] 3번 상공으로 수직 상승
🚀 [3 ➡️ 1] 다시 1번 상공으로 수평 복귀 (X, Y축 복귀)
🎉 [성공] Y축 피드백을 반영하여 직각 및 평면 모션을 안전하게 완수했습니다!


In [15]:
try:
    print("그리퍼 열기")
    gripper_open()
    wait(2)  # ★ 그리퍼가 완전히 열릴 때까지 여유 있게 대기
    
    print("🚀 [시작] 1번 기본 위치로 기동")
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    # --- 1 ➡️ 2 ➡️ 1 구간 (수직 하강 및 상승) ---
    print("🚀 [1 ➡️ 2] 수직 하강 (Z축만 이동)")
    movel(posx(PICK_2), vel=SPEED_L * 0.5, acc=ACC_L * 0.5, ref=DR_BASE) # 진입 시 감속
    wait(1)
    
    print("🔥 그리퍼 상자 압착 시작 (힘 강화)")
    # 헐거움을 방지하기 위해 힘(current)을 예시 코드 수준인 300(또는 최소 200 이상)으로 올립니다.
    gripper_close(current=300)  
    
    # ★ 매우 중요: 그리퍼가 상자를 완전히 조여서 꽉 잡을 때까지 '물리적인 시간'을 충분히 줍니다.
    print("⏳ 그리퍼가 상자를 꽉 쥘 때까지 대기 중...")
    wait(3.5) 

    print("🚀 [2 ➡️ 1] 수직 상승 복귀 (상자를 들고 상승)")
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    # --- 1 ➡️ 3 ➡️ 4 구간 (X, Y축 평면 이동 후 수직 하강) ---
    print("🚀 [1 ➡️ 3] 수평 평면 이동 (X, Y축 동시 이동하여 4번 상공 안착)")
    movel(posx(PICK_3), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    print("🚀 [3 ➡️ 4] 4번 타겟으로 수직 하강")
    movel(posx(PICK_4), vel=SPEED_L * 0.5, acc=ACC_L * 0.5, ref=DR_BASE) # 진입 시 감속
    wait(1)

    print("그리퍼 열기 (상자 내려놓기)")
    gripper_open()
    
    # ★ 중요: 상자를 안전하게 완전히 놓아줄 때까지 대기한 후 로봇팔을 빼야 합니다.
    print("⏳ 그리퍼가 완전히 열릴 때까지 대기 중...")
    wait(3.0) 
    
    # --- 4 ➡️ 3 ➡️ 1 구간 (역순 안전 복귀) ---
    print("🚀 [4 ➡️ 3] 3번 상공으로 수직 상승")
    movel(posx(PICK_3), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    print("🚀 [3 ➡️ 1] 다시 1번 상공으로 수평 복귀 (X, Y축 복귀)")
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

    print("🎉 [성공] Y축 피드백을 반영하여 직각 및 평면 모션을 안전하게 완수했습니다!")

except Exception as e:
    print(f"❌ 구동 중 에러 발생: {e}")

그리퍼 열기
🚀 [시작] 1번 기본 위치로 기동
🚀 [1 ➡️ 2] 수직 하강 (Z축만 이동)
🔥 그리퍼 상자 압착 시작 (힘 강화)
⏳ 그리퍼가 상자를 꽉 쥘 때까지 대기 중...
🚀 [2 ➡️ 1] 수직 상승 복귀 (상자를 들고 상승)
🚀 [1 ➡️ 3] 수평 평면 이동 (X, Y축 동시 이동하여 4번 상공 안착)
🚀 [3 ➡️ 4] 4번 타겟으로 수직 하강
그리퍼 열기 (상자 내려놓기)
⏳ 그리퍼가 완전히 열릴 때까지 대기 중...
🚀 [4 ➡️ 3] 3번 상공으로 수직 상승
🚀 [3 ➡️ 1] 다시 1번 상공으로 수평 복귀 (X, Y축 복귀)
🎉 [성공] Y축 피드백을 반영하여 직각 및 평면 모션을 안전하게 완수했습니다!
